## Libraries

In [ ]:
import torch
import numpy as np
from fedot_ind.core.operation.transformation.torch_backend.tabular.tabular_extractor import PCA_learning_layer

import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score

/workspaces/Fedot.Industrial/.venv/lib/python3.10/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
batch_size = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

## Baseline

## Data

In [ ]:
X, y = fetch_openml("PokerHand", version=1, return_X_y=True, as_frame=False)
X = X.astype(np.float32)
y = y.astype(np.int64)
valid_classes = [0, 1, 2, 3, 4, 5, 6]  # например
mask = np.isin(y, valid_classes)
X, y = X[mask], y[mask]
print(X.shape)
print(y.shape)
torch.backends.cudnn.benchmark = False
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)
print('train:', X_train.shape)
print('test:', X_test.shape)
print('num classes:', len(set(y)))

(828993, 10)
(828993,)
train: torch.Size([663194, 10])
test: torch.Size([165799, 10])
num classes: 7


## Baseline

In [ ]:
class BaseMLP(nn.Module):
    def __init__(self, input_dim=64, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            # nn.Linear(input_dim, 128),
            # nn.ReLU(),
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        return self.net(x)

def create_batches(X, y, batch_size=512):
    """Разбивает X и y на списки батчей"""
    n_samples = X.shape[0]
    batches_X = []
    batches_y = []
    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        batches_X.append(X[start:end])
        batches_y.append(y[start:end])
    return batches_X, batches_y

def base_fit(model, X_train, y_train, epochs, batch_size=512, lr=1e-4, device='cuda'):
    # Переносим модель и все данные на устройство
    model.to(device)
    X_train = X_train.to(device)
    y_train = y_train.to(device)

    # Создаём батчи
    batches_X, batches_y = create_batches(X_train, y_train, batch_size)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch_X, batch_y in zip(batches_X, batches_y):
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(batches_X)

        model.eval()
        with torch.no_grad():
            train_pred = torch.argmax(model(X_train), dim=1)
            train_acc = (train_pred == y_train).float().mean().item()

        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f}")

    return model

In [6]:
#baseline
epochs = 20
model = BaseMLP(input_dim=X_train.shape[1], num_classes=7)
model = base_fit(model, X_train, y_train, epochs)

Epoch 01/20 | Loss: 1.2721 | Train Acc: 0.5800
Epoch 02/20 | Loss: 0.9198 | Train Acc: 0.6227
Epoch 03/20 | Loss: 0.8383 | Train Acc: 0.6754
Epoch 04/20 | Loss: 0.7609 | Train Acc: 0.7166
Epoch 05/20 | Loss: 0.6920 | Train Acc: 0.7440
Epoch 06/20 | Loss: 0.6352 | Train Acc: 0.7647
Epoch 07/20 | Loss: 0.5873 | Train Acc: 0.7827
Epoch 08/20 | Loss: 0.5444 | Train Acc: 0.8017
Epoch 09/20 | Loss: 0.5038 | Train Acc: 0.8219
Epoch 10/20 | Loss: 0.4642 | Train Acc: 0.8437
Epoch 11/20 | Loss: 0.4255 | Train Acc: 0.8654
Epoch 12/20 | Loss: 0.3877 | Train Acc: 0.8869
Epoch 13/20 | Loss: 0.3509 | Train Acc: 0.9067
Epoch 14/20 | Loss: 0.3157 | Train Acc: 0.9241
Epoch 15/20 | Loss: 0.2827 | Train Acc: 0.9384
Epoch 16/20 | Loss: 0.2522 | Train Acc: 0.9500
Epoch 17/20 | Loss: 0.2245 | Train Acc: 0.9598
Epoch 18/20 | Loss: 0.1993 | Train Acc: 0.9675
Epoch 19/20 | Loss: 0.1765 | Train Acc: 0.9738
Epoch 20/20 | Loss: 0.1560 | Train Acc: 0.9788


In [7]:
model.eval()
with torch.no_grad():
    pred = torch.argmax(model(X_test.to(device)), dim=1)
    acc_base = accuracy_score(y_test.cpu(), pred.cpu())
print(acc_base)

0.9780336431462192


## PCA Classifier

In [ ]:
class PCAClassifier(nn.Module):
    def __init__(self, input_dim, explained_variance=0.95, hidden_dim=64, freeze_epochs=5, num_classes=2):
        super().__init__()
        self.pca_layer = PCA_learning_layer(
            input_dim=input_dim,
            explained_variance=explained_variance,
            freeze_epochs=freeze_epochs
        )
        self.hidden_dim = hidden_dim
        self.freeze_epochs = freeze_epochs
        self.num_classes = num_classes
        self.classifier = None 

    def forward(self, X):
        X_pca = self.pca_layer(X)
        return self.classifier(X_pca)

    def build_classifier(self):
        out_dim = self.pca_layer.W.shape[1]
        print(out_dim)
        self.classifier = nn.Sequential(
            # nn.Linear(out_dim, 128),
            # nn.ReLU(),
            nn.Linear(out_dim, 64),
            nn.ReLU(),
            nn.Linear(64, self.num_classes)
        )

    def step_epoch(self):
        self.pca_layer.step_epoch()

    @torch.no_grad()
    def predict(self, X):
        self.eval()
        logits = self.forward(X)
        preds = torch.argmax(logits, dim=1)
        return preds

    @torch.no_grad()
    def evaluate(self, X, y_true):
        y_pred = self.predict(X)
        acc = (y_pred == y_true).float().mean().item()
        return acc



def train_pca_model(model, X_train, y_train, epochs=20, batch_size=64, lr=1e-4, is_loss_ar=False, device='cpu'):
    model.to(device)
    X_train = X_train.to(device)
    y_train = y_train.to(device)

    batches_X, batches_y = create_batches(X_train, y_train, batch_size)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_ar, epoch_grad_norms = [], []

    for epoch in range(epochs):
        model.train()
        total_loss, total_norm = 0, 0
        # if epoch == model.freeze_epochs:
        #     optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
        #     print(f"[INFO] Unfreezing PCA weights at epoch {epoch}")
        for batch_X, batch_y in zip(batches_X, batches_y):
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y.long().squeeze())
            loss.backward()
            total_norm = torch.norm(
                torch.stack([p.grad.detach().norm(2) for p in model.parameters() if p.grad is not None])
            )
            optimizer.step()
            total_loss += loss.item()
        model.step_epoch()
        avg_loss = total_loss / len(batches_X)
        avg_grad_norm = total_norm / len(batches_X)
        loss_ar.append(avg_loss)
        epoch_grad_norms.append(avg_grad_norm)
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | GradNorm: {avg_grad_norm:.4f}")
    if is_loss_ar:
        return model, loss_ar, epoch_grad_norms
    return model

In [33]:
epochs=50
model = PCAClassifier(input_dim=X_train.shape[1], num_classes=10, freeze_epochs=10, explained_variance=0.95)
model = model.to(device)
model.pca_layer.fit(X_train)
model.build_classifier()

trained_model, losses, grad_norms = train_pca_model(
    model, X_train, y_train, epochs=50, batch_size=256, is_loss_ar=True, lr=1e-3
)
model.eval()
trained_model.to(device)
with torch.no_grad():
    y_pred = trained_model.predict(X_test.to(device))
    acc_pca = accuracy_score(y_test.cpu(), y_pred.cpu())
print(acc_pca)

10
Epoch 01/50 | Loss: 1.0040 | GradNorm: 0.0002
Epoch 02/50 | Loss: 0.9727 | GradNorm: 0.0002
Epoch 03/50 | Loss: 0.9718 | GradNorm: 0.0002
Epoch 04/50 | Loss: 0.9714 | GradNorm: 0.0002
Epoch 05/50 | Loss: 0.9712 | GradNorm: 0.0002
Epoch 06/50 | Loss: 0.9711 | GradNorm: 0.0002
Epoch 07/50 | Loss: 0.9710 | GradNorm: 0.0002
Epoch 08/50 | Loss: 0.9709 | GradNorm: 0.0002
Epoch 09/50 | Loss: 0.9708 | GradNorm: 0.0002
Epoch 10/50 | Loss: 0.9707 | GradNorm: 0.0002
Epoch 11/50 | Loss: 0.9706 | GradNorm: 0.0003
Epoch 12/50 | Loss: 0.9706 | GradNorm: 0.0006
Epoch 13/50 | Loss: 0.9705 | GradNorm: 0.0009
Epoch 14/50 | Loss: 0.9705 | GradNorm: 0.0012
Epoch 15/50 | Loss: 0.9704 | GradNorm: 0.0014
Epoch 16/50 | Loss: 0.9704 | GradNorm: 0.0017
Epoch 17/50 | Loss: 0.9704 | GradNorm: 0.0020
Epoch 18/50 | Loss: 0.9703 | GradNorm: 0.0022
Epoch 19/50 | Loss: 0.9703 | GradNorm: 0.0025
Epoch 20/50 | Loss: 0.9703 | GradNorm: 0.0028
Epoch 21/50 | Loss: 0.9703 | GradNorm: 0.0030
Epoch 22/50 | Loss: 0.9703 | Gr

## PCA layer with different stop criterions

In [ ]:
def create_batches(X, y, batch_size):
    n = X.size(0)
    indices = torch.randperm(n)
    X, y = X[indices], y[indices]
    batches_X = [X[i:i+batch_size] for i in range(0, n, batch_size)]
    batches_y = [y[i:i+batch_size] for i in range(0, n, batch_size)]
    return batches_X, batches_y


def train_pca_model(
    model,
    X_train,
    y_train,
    epochs=50,
    batch_size=64,
    lr=1e-3,
    freeze_epoch=None,
    unfreeze_mode="auto",   # 'grad_norm', 'loss', 'epoch', 'auto'
    grad_tolerance=1e-4,
    loss_tolerance=1e-3,
    device="cuda",
    is_loss_ar=False
):
    """
    unfreeze_mode:
        'grad_norm': разморозка, когда изменение нормы градиента < grad_tolerance
        - 'loss': разморозка, когда изменение лосса < loss_tolerance
        - 'epoch': разморозка по фиксированному количеству эпох
        - 'auto': freeze_epochs = 10% от epochs
    """
    model.to(device)
    X_train, y_train = X_train.to(device), y_train.to(device)
    batches_X, batches_y = create_batches(X_train, y_train, batch_size)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    if freeze_epoch is not None:
        model.pca_layer.freeze_epochs = freeze_epoch
    else:
        if unfreeze_mode == "auto":
            model.pca_layer.freeze_epochs = max(1, int(0.1 * epochs))

    loss_ar, epoch_grad_norms = [], []
    prev_loss, prev_grad_norm = None, None
    pca_unfrozen = False

    for epoch in range(epochs):
        model.train()
        total_loss, total_norm = 0.0, 0.0

        for batch_X, batch_y in zip(batches_X, batches_y):
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y.long().squeeze())
            loss.backward()
            total_norm = torch.norm(
                torch.stack([p.grad.detach().norm(2)
                             for p in model.parameters()
                             if p.grad is not None])
            ).item()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(batches_X)
        avg_grad_norm = total_norm / len(batches_X)
        loss_ar.append(avg_loss)
        epoch_grad_norms.append(avg_grad_norm)

        if not pca_unfrozen:
            if unfreeze_mode == "grad_norm" and prev_grad_norm is not None:
                if abs(prev_grad_norm - avg_grad_norm) < grad_tolerance:
                    model.pca_layer.unfreeze_weights()
                    pca_unfrozen = True
                    print(f"Unfreezing PCA weights at epoch {epoch} (grad_norm stabilized)")
            elif unfreeze_mode == "loss" and prev_loss is not None:
                if abs(prev_loss - avg_loss) < loss_tolerance:
                    model.pca_layer.unfreeze_weights()
                    pca_unfrozen = True
                    print(f"Unfreezing PCA weights at epoch {epoch} (loss stabilized)")
            elif unfreeze_mode in ("epoch", "auto") and epoch == model.pca_layer.freeze_epochs:
                model.pca_layer.unfreeze_weights()
                pca_unfrozen = True
                print(f"Unfreezing PCA weights at epoch {epoch} (fixed freeze_epochs={model.pca_layer.freeze_epochs})")

        prev_loss = avg_loss
        prev_grad_norm = avg_grad_norm
        if epoch % 20 == 0:
            print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | GradNorm: {avg_grad_norm:.4f}")

        model.step_epoch()

    if is_loss_ar:
        return model, loss_ar, epoch_grad_norms
    return model


## Tests

In [ ]:
epochs=150
freeze_epochs = [None] #100, 10, 20, 30, 
modes = ['loss'] #'auto', 'epoch'
acc = []
f1 = []
precision = []

for fe in freeze_epochs:
    model = PCAClassifier(input_dim=X_train.shape[1], num_classes=7, freeze_epochs=10, explained_variance=0.9)
    model = model.to(device)
    model.pca_layer.fit(X_train)
    model.build_classifier()
    if fe is None:
        for mode in modes:
            model = PCAClassifier(input_dim=X_train.shape[1], num_classes=7, freeze_epochs=10, explained_variance=0.9)
            model = model.to(device)
            model.pca_layer.fit(X_train)
            model.build_classifier()
            trained_model = train_pca_model(
                model, X_train, y_train, epochs=epochs, batch_size=256, is_loss_ar=False, lr=1e-3,
                unfreeze_mode=mode
            )
            trained_model.eval()
            trained_model.to(device)
            with torch.no_grad():
                y_pred = trained_model.predict(X_test.to(device))
                acc_pca = accuracy_score(y_test.cpu(), y_pred.cpu())
                f1_pca = f1_score(y_test.cpu(), y_pred.cpu(), average='weighted')
                precision_pca = precision_score(y_test.cpu(), y_pred.cpu(), average='weighted')
                acc.append(acc_pca)
                f1.append(f1_pca)
                precision.append(precision_pca)

            print(f"freeze_epoch: {fe}  mode: {mode}")
            print(f"acc: {acc_pca:.4f} f1: {f1_pca:.4f} precision: {precision_pca:.4f}", "\n")
    else:
        trained_model = train_pca_model(
            model, X_train, y_train, epochs=epochs, batch_size=256, is_loss_ar=False, lr=1e-3,
            unfreeze_mode='auto', freeze_epoch=fe
        )
        trained_model.eval()
        trained_model.to(device)
        with torch.no_grad():
            y_pred = trained_model.predict(X_test.to(device))
            acc_pca = accuracy_score(y_test.cpu(), y_pred.cpu())
            f1_pca = f1_score(y_test.cpu(), y_pred.cpu(), average='weighted')
            precision_pca = precision_score(y_test.cpu(), y_pred.cpu(), average='weighted')
            acc.append(acc_pca)
            f1.append(f1_pca)
            precision.append(precision_pca)

        print(f"freeze_epoch: {fe}  mode: auto")
        print(f"acc: {acc_pca:.4f} f1: {f1_pca:.4f} precision: {precision_pca:.4f}", "\n")

7
Epoch 01/150 | Loss: 0.8697 | GradNorm: 0.0002
Epoch 21/150 | Loss: 0.6845 | GradNorm: 2.2336
Epoch 41/150 | Loss: 0.6253 | GradNorm: 8.5959
Epoch 61/150 | Loss: 0.5836 | GradNorm: 17.7789
Epoch 81/150 | Loss: 0.5133 | GradNorm: 29.6054
Epoch 101/150 | Loss: 0.4704 | GradNorm: 43.7893
[INFO] Unfreezing PCA weights at epoch 120 (loss stabilized)
Epoch 121/150 | Loss: 0.4469 | GradNorm: 63.0579
Epoch 141/150 | Loss: 0.4284 | GradNorm: 86.2884
freeze_epoch: None  mode: loss
acc: 0.8225 f1: 0.8147 precision: 0.8197 

